# 🎵 ACE-Step 1.5 no Google Colab
Gerador de música open source (estilo Suno). Rode as células **em ordem**, uma de cada vez.

**Antes de começar:** menu *Ambiente de execução → Alterar tipo de ambiente de execução → GPU* (A100 para qualidade máxima, T4/L4 para testar).

## 1) Verificar a GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2) Instalar o ACE-Step (demora alguns minutos)

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

!git clone https://github.com/ACE-Step/ACE-Step-1.5.git
%cd ACE-Step-1.5
!uv sync

## 3) Escolher os modelos
- **A100 / GPU de 16 GB+**: modelo de linguagem `4B` (melhor composição)
- **T4 (16 GB) ou menos**: troque para `1.7B` na variável abaixo

In [ ]:
LM_MODEL = "acestep-5Hz-lm-4B"   # troque para "acestep-5Hz-lm-1.7B" se estiver numa T4

env = f"""ACESTEP_CONFIG_PATH=acestep-v15-turbo
ACESTEP_LM_MODEL_PATH={LM_MODEL}
PORT=7860
LANGUAGE=en
"""
open(".env", "w").write(env)
print(env)

## 4) Ligar a IA em segundo plano (os modelos baixam na primeira vez)

In [ ]:
import subprocess, time, urllib.request

proc = subprocess.Popen("uv run acestep --port 7860 > /content/acestep.log 2>&1", shell=True)

for i in range(120):   # espera até ~10 min
    try:
        urllib.request.urlopen("http://localhost:7860", timeout=3)
        print("✅ ACE-Step no ar!")
        break
    except Exception:
        time.sleep(5)
else:
    print("⚠️ Ainda não subiu. Veja o log na próxima célula.")

## 5) (Só se precisar) Ver o log para diagnosticar erros

In [ ]:
!tail -n 40 /content/acestep.log

## 6) Gerar o link para abrir no navegador
Procure na saída um endereço terminado em **trycloudflare.com** e abra. 
Deixe esta célula rodando enquanto estiver usando.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
!/content/cloudflared tunnel --url http://localhost:7860

## Dicas
- **Baixe as músicas** assim que gerar: quando o Colab desconecta, tudo é apagado.
- Ao terminar: *Ambiente de execução → Desconectar e excluir ambiente de execução* para não gastar unidades à toa.